In [1]:
import numpy as np
import pandas as pd

In [2]:
# Preparing the dataset

df = pd.read_csv("./data.csv")

df['horsepower'] = df['horsepower'].fillna(0)
df['acceleration'] = df['acceleration'].fillna(0)
df.dtypes

model_year               int64
origin                     str
fuel_type                  str
drivetrain                 str
num_doors                int64
engine_displacement      int64
num_cylinders            int64
horsepower             float64
vehicle_weight           int64
acceleration           float64
fuel_efficiency_mpg    float64
dtype: object

In [3]:
from sklearn.model_selection import train_test_split


df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

In [4]:
df_full_train = df_full_train.reset_index(drop=True)
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_full_train = df_full_train['fuel_efficiency_mpg']
y_train = df_train['fuel_efficiency_mpg']
y_val = df_val['fuel_efficiency_mpg']
y_test = df_test['fuel_efficiency_mpg']

del df_full_train['fuel_efficiency_mpg']
del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

In [5]:
from sklearn.feature_extraction import DictVectorizer


dv = DictVectorizer(sparse=True)

train_dicts = df_train.to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val.to_dict(orient='records')
X_val = dv.transform(val_dicts)

test_dicts = df_test.to_dict(orient='records')
X_test = dv.transform(test_dicts)

full_train_dicts = df_full_train.to_dict(orient='records')
X_full_train = dv.transform(full_train_dicts)

In [6]:
# Question 1: 'model_year'

from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text


dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)

print(export_text(dt, feature_names=list(dv.get_feature_names_out())))

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



In [7]:
def rmse(y, y_pred):
    se = (y - y_pred) ** 2
    mse = se.mean()
    return np.sqrt(mse)

In [8]:
# Question 2: 1.839
from sklearn.ensemble import RandomForestRegressor


rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)

rmse(y_val, y_pred)

np.float64(1.8393827905033797)

In [9]:
# Question 3: 100

for n in [10, 50, 100, 150]:
    rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_val)
    print(f'{n}: {rmse(y_val, y_pred)}')

10: 1.8393827905033797
50: 1.7728631357214235
100: 1.7676431541745072
150: 1.768109448598198


In [10]:
# Question 4: 10

for d in [10, 15, 20, 25]:
    rmse_scores = []
    for n in [10, 50, 100, 150]:
        rf = RandomForestRegressor(max_depth=d, n_estimators=n, random_state=1, n_jobs=-1)
        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_val)
        rmse_scores.append(rmse(y_val, y_pred))
    print(f'{d}: {sum(rmse_scores) / len(rmse_scores)}')


10: 1.754133023722073
15: 1.7836702888451024
20: 1.7878464776552963
25: 1.7866745521668288


In [11]:
# Question 5: vehicle_weight

rf = RandomForestRegressor(max_depth=20, n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

importance = pd.DataFrame({
    'feature': dv.get_feature_names_out(),
    'importance': rf.feature_importances_
})

print(importance)

                         feature  importance
0                   acceleration    0.055342
1     drivetrain=All-wheel drive    0.007971
2   drivetrain=Front-wheel drive    0.020108
3    drivetrain=Rear-wheel drive    0.005574
4            engine_displacement    0.061152
5               fuel_type=Diesel    0.018045
6             fuel_type=Gasoline    0.160712
7               fuel_type=Hybrid    0.014831
8                     horsepower    0.078468
9                     model_year    0.295915
10                 num_cylinders    0.007227
11                     num_doors    0.019364
12                   origin=Asia    0.006388
13                 origin=Europe    0.003566
14                    origin=USA    0.042268
15                vehicle_weight    0.203067


In [12]:
import xgboost as xgb

In [13]:
# Question 6: 0.1

features = list(dv.get_feature_names_out())
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)

watchlist = [(dtrain, 'train'), (dval, 'val')]

eta_scores = []

for e in [0.3, 0.1]:
    xgb_params = {
        'eta': e, 
        'max_depth': 6,
        'min_child_weight': 1,
        
        'objective': 'reg:squarederror',
        'nthread': 8,
        
        'seed': 1,
        'verbosity': 1,
    }

    model = xgb.train(xgb_params, dtrain, num_boost_round=100, evals=watchlist)

    y_pred = model.predict(dval)
    eta_scores.append(f'{e}: {rmse(y_val, y_pred)}')

eta_scores

[0]	train-rmse:2.40653	val-rmse:2.46782
[1]	train-rmse:2.07098	val-rmse:2.15784
[2]	train-rmse:1.86454	val-rmse:1.97398
[3]	train-rmse:1.73360	val-rmse:1.86662
[4]	train-rmse:1.65059	val-rmse:1.80312
[5]	train-rmse:1.59328	val-rmse:1.76216
[6]	train-rmse:1.55435	val-rmse:1.74392
[7]	train-rmse:1.52677	val-rmse:1.73976
[8]	train-rmse:1.50265	val-rmse:1.73478
[9]	train-rmse:1.48761	val-rmse:1.73282
[10]	train-rmse:1.47125	val-rmse:1.73205
[11]	train-rmse:1.45902	val-rmse:1.73227
[12]	train-rmse:1.44829	val-rmse:1.73503
[13]	train-rmse:1.43604	val-rmse:1.73466
[14]	train-rmse:1.42972	val-rmse:1.73448
[15]	train-rmse:1.41908	val-rmse:1.73512
[16]	train-rmse:1.41206	val-rmse:1.73297
[17]	train-rmse:1.40940	val-rmse:1.73517
[18]	train-rmse:1.40278	val-rmse:1.73584
[19]	train-rmse:1.39637	val-rmse:1.73664
[20]	train-rmse:1.39368	val-rmse:1.73766
[21]	train-rmse:1.39254	val-rmse:1.73701
[22]	train-rmse:1.37639	val-rmse:1.74216
[23]	train-rmse:1.36470	val-rmse:1.74363
[24]	train-rmse:1.36071	va

['0.3: 1.8264579799904583', '0.1: 1.724810059827976']